# T08: Fabric SQL Database Quick Start

## What You'll Learn

Microsoft Fabric SQL Database provides a fully managed SQL Server experience. In this notebook you will:

1. **Generate** a retail dataset at small scale
2. **Export** as SQL files containing DDL and INSERT statements
3. **Inspect** the generated SQL output
4. **Understand** how to use these scripts against a real Fabric SQL Database

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle`

## Time Estimate

**~5 minutes** from start to finish.

In [ ]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Generate Retail Data

### What we're about to do
We'll generate a complete retail dataset using `small` scale. This produces realistic customers, products, orders, and order items with valid foreign-key relationships.

### Why this matters
Before you can load data into Fabric SQL Database, you need data that respects relational constraints. Spindle guarantees referential integrity out of the box, so you can focus on testing your loading pipeline rather than debugging bad data.

### What to expect
A summary showing every table and its row count.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())

## Export as SQL INSERT Files

### What we're about to do
We'll call `result.to_sql()` to produce SQL scripts. Each table gets its own `.sql` file containing:
- A `CREATE TABLE` statement with proper column types and constraints
- Batched `INSERT INTO` statements (100 rows per batch by default)

### Why this matters
SQL scripts are the most portable way to seed a database. You can run them in Azure Data Studio, SSMS, sqlcmd, or a Fabric notebook. Batching INSERTs avoids hitting query-size limits.

### What to expect
A list of `.sql` files written to the output directory.

In [ ]:
files = result.to_sql("./sql_output/", schema_name="dbo", batch_size=100)

print(f"Wrote {len(files)} SQL files:\n")
import os
for f in files:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f)} ({size:,} bytes)")

## Inspect the SQL Output

### What we're about to do
We'll read the first SQL file and display the first 2,000 characters. This lets you see the DDL (CREATE TABLE) and the beginning of the INSERT statements.

### Why this matters
Inspecting the output confirms that column types are correct, constraints are present, and the INSERT syntax matches what your target database expects. This is especially important when targeting Fabric SQL Database, which has some T-SQL nuances.

### What to expect
A `CREATE TABLE` statement followed by `INSERT INTO ... VALUES` batches.

In [ ]:
print(f"=== Preview: {os.path.basename(files[0])} ===")
print()
with open(files[0]) as f:
    print(f.read()[:2000])
print("\n... (truncated)")

## Inspect a Second File

### What we're about to do
We'll preview a second SQL file to see how different table structures are handled — different column types, foreign-key constraints, and INSERT batches.

### Why this matters
Seeing multiple tables confirms that Spindle correctly maps each domain's column types (strings, decimals, dates, etc.) to valid T-SQL types.

### What to expect
Another CREATE TABLE + INSERT preview, this time for a different table.

In [ ]:
if len(files) > 1:
    print(f"=== Preview: {os.path.basename(files[1])} ===")
    print()
    with open(files[1]) as f:
        print(f.read()[:2000])
    print("\n... (truncated)")
else:
    print("Only one SQL file was generated.")

## Using These Scripts with Fabric SQL Database

### Connection String Pattern

To run these scripts against a real Fabric SQL Database, you'll need a connection string. In Fabric, the SQL endpoint looks like:

```
Server=<your-workspace>.datawarehouse.fabric.microsoft.com
Database=<your-sql-database>
Authentication=ActiveDirectoryInteractive
```

### Execution Options

| Tool | How |
|------|-----|
| **Fabric Notebook** | Upload `.sql` files and run with `%%sql` magic or `pyodbc` |
| **Azure Data Studio** | Connect with AAD auth, open script, execute |
| **sqlcmd** | `sqlcmd -S <server> -d <db> -G -i customers.sql` |
| **Python (pyodbc)** | Read the `.sql` file and execute via `cursor.execute()` |

### Important Notes

- Execute parent tables **before** child tables to satisfy foreign-key constraints
- The `schema_name` parameter defaults to `dbo` — change it if your Fabric database uses a different schema
- Batch size of 100 keeps each INSERT under Fabric's query-size limits

In [ ]:
# Example: reading a SQL file for programmatic execution
# (This cell shows the pattern — it does NOT connect to a real database)

sql_file = files[0]
with open(sql_file) as f:
    sql_content = f.read()

# Count the number of statements
statements = [s.strip() for s in sql_content.split(";") if s.strip()]
print(f"File: {os.path.basename(sql_file)}")
print(f"Total statements: {len(statements)}")
print(f"DDL statements: {sum(1 for s in statements if s.upper().startswith('CREATE'))}")
print(f"INSERT statements: {sum(1 for s in statements if s.upper().startswith('INSERT'))}")

print("\n# To execute against Fabric SQL Database:")
print("# import pyodbc")
print("# conn = pyodbc.connect(connection_string)")
print("# cursor = conn.cursor()")
print("# for stmt in statements:")
print("#     cursor.execute(stmt)")
print("# conn.commit()")

## What's Next?

You've generated a retail dataset and exported it as SQL scripts ready for Fabric SQL Database. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T07: Fabric Lakehouse Quick Start** | Write Parquet and Delta Lake files for Fabric Lakehouse |
| **T09: Streaming Events** | Stream real-time events with configurable throughput |
| **T10: File-Drop Simulation** | Simulate daily file drops with manifests and done flags |

Happy generating!